# UC Atlas: Cell Type Abundance Analysis (Fig. 1E)

Kruskal-Wallis H-test to assess condition-driven differences in cell type
abundance across UC samples (healthy, non-inflamed, inflamed).

For each cell type, per-sample cell type fractions are compared across
the three conditions using the non-parametric Kruskal-Wallis test.
Results are visualized as boxplots with significance annotation.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
OUTPUT_DIR = f"{DATA_DIR}/abundance"

In [ ]:
# Load annotated atlas
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")

## 1. Compute per-sample cell type fractions

In [ ]:
# Count cells per sample × cell type
counts = (
    scrna.obs
    .groupby(["sample", "condition", "cell_type_general"])
    .size()
    .reset_index(name="n_cells")
)

# Total cells per sample
totals = counts.groupby("sample")["n_cells"].sum().rename("n_total")
counts = counts.merge(totals, on="sample")
counts["fraction"] = counts["n_cells"] / counts["n_total"]

counts.head()

## 2. Kruskal-Wallis H-test per cell type

In [ ]:
conditions = ["Healthy", "Non-inflamed", "Inflamed"]
cell_types = counts["cell_type_general"].unique()

kw_results = []
for ct in cell_types:
    ct_data = counts[counts["cell_type_general"] == ct]
    groups = [
        ct_data.loc[ct_data["condition"] == cond, "fraction"].values
        for cond in conditions
        if cond in ct_data["condition"].values
    ]
    if len(groups) >= 2 and all(len(g) > 0 for g in groups):
        stat, pval = kruskal(*groups)
        kw_results.append({"cell_type": ct, "H_stat": stat, "pval": pval})

kw_df = pd.DataFrame(kw_results)

# FDR correction (Benjamini-Hochberg)
_, kw_df["padj"], _, _ = multipletests(kw_df["pval"], method="fdr_bh")
kw_df = kw_df.sort_values("padj")

kw_df.to_csv(f"{OUTPUT_DIR}/kruskal_wallis_results.csv", index=False)
kw_df

## 3. Visualize abundance by condition (Fig. 1E)

In [ ]:
# Significant cell types (FDR < 0.05)
sig_cts = kw_df.loc[kw_df["padj"] < 0.05, "cell_type"].tolist()

# Order cell types by median inflamed fraction
inflamed_medians = (
    counts[counts["condition"] == "Inflamed"]
    .groupby("cell_type_general")["fraction"]
    .median()
    .sort_values(ascending=False)
)
ct_order = inflamed_medians.index.tolist()

plot_data = counts[counts["cell_type_general"].isin(ct_order)].copy()
plot_data["cell_type_general"] = pd.Categorical(
    plot_data["cell_type_general"], categories=ct_order
)

In [ ]:
palette = {"Healthy": "#4dac26", "Non-inflamed": "#f1b64a", "Inflamed": "#d01c8b"}
n_ct = len(ct_order)

fig, axes = plt.subplots(1, n_ct, figsize=(n_ct * 1.2, 4), sharey=False)
if n_ct == 1:
    axes = [axes]

for ax, ct in zip(axes, ct_order):
    ct_data = plot_data[plot_data["cell_type_general"] == ct]
    sns.boxplot(
        data=ct_data, x="condition", y="fraction",
        order=conditions, palette=palette,
        width=0.5, showcaps=False, flierprops=dict(marker=""),
        ax=ax
    )
    sns.stripplot(
        data=ct_data, x="condition", y="fraction",
        order=conditions, palette=palette,
        size=2.5, jitter=True, alpha=0.5, ax=ax
    )
    # Significance asterisk
    row = kw_df[kw_df["cell_type"] == ct]
    if not row.empty and row.iloc[0]["padj"] < 0.05:
        ymax = ct_data["fraction"].max()
        ax.text(1, ymax * 1.05, "*", ha="center", fontsize=12)

    ax.set_title(ct, fontsize=7, rotation=45, ha="right")
    ax.set_xlabel("")
    ax.set_ylabel("Fraction" if ct == ct_order[0] else "")
    ax.set_xticks([])
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle("Cell Type Abundance by Condition (UC)", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1e_cell_type_abundance_kruskal_wallis.pdf",
            bbox_inches="tight", dpi=300)
plt.show()